# TFP Swap Spread Packages V2 — OU Exits, PCA Weighting, Full Universe

Trades all 15 curves + 20 flies from the TFP deviation term structure, with OU mean-reversion exits and optional PCA-weighted sizing.

**Universe:** All C(6,2)=15 curve pairs and C(6,3)=20 butterfly flies from the regression tenor set \[2Y, 3Y, 5Y, 7Y, 10Y, 30Y\].

**Signal:** Rolling z-score of the deviation differential, with OU-calibrated exit (crosses long-run mean) or fixed z-score exit.

**Optional PCA weighting:** Ledoit-Wolf shrinkage covariance → inverse-vol / GMV / MSR position sizing.

In [ ]:
%load_ext autoreload
%autoreload 2

import datetime, sys, os, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import QuantLib as ql

plt.style.use('ggplot')
pylab.rcParams.update({
    'legend.fontsize': 'x-large', 'figure.figsize': (18, 8),
    'axes.labelsize': 'x-large', 'axes.titlesize': 'x-large',
    'xtick.labelsize': 'large', 'ytick.labelsize': 'large',
})

sys.path.append('../../')

from BT.data_handler import TimeGrid
from BT.misc import ql_cal_date_range
from BT.query_actions import AddQueryAction, UnwindPositionsAction
from BT.query_engine import QueryDrivenBacktest
from BT.query_strategy import QueryStrategy
from BT.triggers import DateTrigger, DateTriggerRequirements
from BT.query_tearsheet import QueryBacktestTearSheet

from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.FixedRateBonds.FixedRateBondQuery import FixedRateBondQuery
from Query.FixedRateBonds.FixedRateBondValue import FixedRateBondValue
from Query.FixedRateBonds.carry_roll import load_us_treasury_gc_fixing_pct
from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapValue import IRSwapValue

from BT.signals.tfp_swap_spread import BENCHMARK_TENORS, REGRESSION_TENORS, CT_MAP, build_tfp_history
from BT.signals.tfp_packages import (
    generate_universe, compute_package_signals, compute_pca_weights,
    extract_package_events, vectorized_preview,
)
from RVUtils.plt_timeseries import make_secondary_axis_plot

## Config

In [ ]:
bt_config = dict(
    signal_start    = datetime.date(2020, 6, 1),
    bt_start        = datetime.date(2021, 6, 1),
    bt_end          = datetime.date(2026, 5, 14),
    z_window        = 60,
    z_entry         = 1.5,
    ou_exit         = True,
    ou_window       = 252,
    fixed_z_exit    = 0.5,
    max_hold        = 60,
    use_pca_weights = False,
    pca_method      = 'inverse_vol',
    pca_cov_window  = 90,
    top_n           = 5,
    risk_bpv        = 100_000,
    unwind_fee_bps  = 0.5,
    specialness_bps = 10.0,
    irs_source      = 'ERIS_EOD_LIVE-RL_BASIC',
    frb_source      = 'USTS_FEDINVEST_WSJ_LIVE-QL',
    curve_name      = 'USD-SOFR-1D',
    cache_path      = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'BT', 'results', 'tfp_screener', 'tfp_history.parquet')),
)
cfg = bt_config
for k, v in cfg.items(): print(f'  {k}: {v}')

## Load TFP History & Generate Universe

In [ ]:
curve_mdp = IRSwapsMDP(source=cfg['irs_source'])
usts_mdp = FixedRateBondsMDP(source='USTS_FEDINVEST_WSJ_LIVE-RL')
history = build_tfp_history(
    start_date=cfg['signal_start'], end_date=cfg['bt_end'],
    curve_mdp=curve_mdp, usts_mdp=usts_mdp,
    cache_path=cfg['cache_path'], show_progress=True,
)
universe = generate_universe()
print(f'TFP history: {len(history)} days')
print(f'Universe: {len(universe)} packages ({len([p for p in universe if p.kind=="curve"])} curves, {len([p for p in universe if p.kind=="fly"])} flies)')

## Compute Signals & Vectorized Preview

In [ ]:
dev_diffs, zscores, signals = compute_package_signals(
    history, universe,
    z_window=cfg['z_window'], z_entry=cfg['z_entry'],
    ou_exit=cfg['ou_exit'], ou_window=cfg['ou_window'],
    fixed_z_exit=cfg['fixed_z_exit'], max_hold=cfg['max_hold'],
)

pca_weights = None
if cfg['use_pca_weights']:
    pca_weights = compute_pca_weights(
        dev_diffs, cov_window=cfg['pca_cov_window'], method=cfg['pca_method'],
    )
    print(f'PCA weights: {pca_weights.shape}')

preview = vectorized_preview(
    history, universe, signals, dev_diffs,
    cfg['bt_start'], cfg['bt_end'], pca_weights=pca_weights,
)
print(f'\nTop 20 packages by Sharpe:')
preview.head(20)

In [ ]:
h_bt = history.loc[cfg['bt_start']:cfg['bt_end']]
top_pkgs = preview.head(10)['Package'].tolist()

plot, fig, ax, ax2, legend = make_secondary_axis_plot(
    ylabel_left='Cumulative P&L (bp)', title=f'Top 10 Packages \u2014 {"OU" if cfg["ou_exit"] else "Fixed"} Exit',
)
for pkg in universe:
    if pkg.name not in top_pkgs:
        continue
    sig = signals[pkg.name].loc[cfg['bt_start']:cfg['bt_end']].reindex(h_bt.index).fillna(0)
    d_mmss = sum(w * h_bt[f'mmss_{leg}'].diff() for w, leg in zip(pkg.weights, pkg.legs) if f'mmss_{leg}' in h_bt.columns)
    cum = (-sig * d_mmss).fillna(0).cumsum()
    plot(cum.rename(pkg.name), which='left', indicators=[{'kind': 'last', 'hide': True}])
legend(show_date=True)
plt.show()

## QueryDrivenBacktest — Top N Packages

In [ ]:
top_names = preview.head(cfg['top_n'])['Package'].tolist()
pkg_map = {p.name: p for p in universe}

all_events = []
for name in top_names:
    pkg = pkg_map[name]
    evts = extract_package_events(signals, pkg, cfg['bt_start'], cfg['bt_end'])
    all_events.extend(evts)
    print(f'  {name}: {len(evts)} trades')
print(f'Total: {len(all_events)} trades')

In [ ]:
CAL = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
tg = TimeGrid(ql_cal_date_range(ql_cal=CAL, start=cfg['bt_start'], end=cfg['bt_end']))

gc_fixing_pct = pd.Series(
    {d: load_us_treasury_gc_fixing_pct(d) for d in sorted({ts.date() for ts in tg})},
    dtype=float,
).sort_index().ffill().bfill()

financing_config = {
    'mode': 'gc_plus_specialness',
    'gc_rate': gc_fixing_pct / 100.0,
    'leg_specialness_bps': {'outright': cfg['specialness_bps']},
    'day_count': 'ACT/360',
    'haircut': 0.0,
}

frb_mdp = FixedRateBondsMDP(source=cfg['frb_source'])
irs_mdp = IRSwapsMDP(source=cfg['irs_source'])

triggers = []
for ev in all_events:
    tag = ev['tag']
    direction = ev['direction']
    pkg = pkg_map[ev['pkg']]
    legs = ev['legs']
    weights = ev['weights']

    entry_actions = []
    for leg_tenor, w in zip(legs, weights):
        ct = CT_MAP[leg_tenor]
        spread_dir = direction * np.sign(w)
        leg_bpv = cfg['risk_bpv'] * spread_dir

        bond_q = FixedRateBondQuery(
            cusip=ct, value=FixedRateBondValue.NPV,
            structure_kwargs={'bpv': leg_bpv},
            meta={'financing': financing_config}, tags=(tag,),
        )
        swap_q = IRSwapQuery(
            curve=cfg['curve_name'], tenor=leg_tenor, value=IRSwapValue.NPV,
            structure_kwargs={'bpv': -leg_bpv}, tags=(tag,),
        )
        entry_actions.extend([AddQueryAction(query=bond_q), AddQueryAction(query=swap_q)])

    triggers.append(DateTrigger(
        DateTriggerRequirements(dates=[ev['entry_date']]),
        actions=entry_actions,
    ))
    triggers.append(DateTrigger(
        DateTriggerRequirements(dates=[ev['exit_date']]),
        actions=[UnwindPositionsAction(
            match_tag=tag, fee=cfg['unwind_fee_bps'] * cfg['risk_bpv'] * len(pkg.legs),
        )],
    ))

print(f'Triggers: {len(triggers)}')

In [ ]:
strategy = QueryStrategy(
    name=f"TFP Packages V2: {', '.join(top_names)}",
    triggers=triggers,
    mdps={'FRB': frb_mdp, 'IRS': irs_mdp},
)
bt = QueryDrivenBacktest(time_grid=tg, strategy=strategy)
t0 = time.time()
bt.run()
print(f'Elapsed: {time.time()-t0:.0f}s')

## Results

In [ ]:
mtm = pd.Series(bt.mtm_history).sort_index()
daily_pnl = mtm.diff().dropna()

plot, fig, ax, ax2, legend = make_secondary_axis_plot(
    ylabel_left='Cumulative P&L', title=strategy.name,
)
plot(mtm.rename('MTM P&L'), which='left', indicators=[
    {'kind': 'last', 'style': {'linestyle': '--', 'linewidth': 1.2}},
])
legend(show_date=True)
plt.show()

sharpe = daily_pnl.mean() / daily_pnl.std() * np.sqrt(252) if daily_pnl.std() > 0 else 0
max_dd = (mtm - mtm.cummax()).min()

print(f'Packages:     {top_names}')
print(f'Signal:       z_entry={cfg["z_entry"]}, ou_exit={cfg["ou_exit"]}, ou_window={cfg["ou_window"]}')
print(f'PCA weights:  {cfg["use_pca_weights"]} ({cfg["pca_method"]})')
print(f'\nFinal MTM:    {mtm.iloc[-1]:>12,.0f}')
print(f'Peak:         {mtm.max():>12,.0f}')
print(f'Max DD:       {max_dd:>12,.0f}')
print(f'Sharpe:       {sharpe:>12.2f}')
print(f'Daily vol:    {daily_pnl.std():>12,.0f}')

In [ ]:
closed = pd.DataFrame(bt.portfolio.closed_positions_log)

if not closed.empty:
    def _extract_tag(row):
        sq = row.get('source_query', None)
        if sq is not None and hasattr(sq, 'tags') and sq.tags:
            return sq.tags[0]
        pos = row.get('position', None)
        if pos is not None and hasattr(pos, 'source_query'):
            sq2 = pos.source_query
            if hasattr(sq2, 'tags') and sq2.tags:
                return sq2.tags[0]
        meta = row.get('position_meta', {})
        if isinstance(meta, dict) and 'tags' in meta and meta['tags']:
            return meta['tags'][0]
        return None

    closed['tag'] = closed.apply(_extract_tag, axis=1)
    cw = closed.dropna(subset=['tag'])

    if not cw.empty:
        tp = cw.groupby('tag').agg(
            pnl=('realized_pnl', 'sum'), days=('holding_period_days', 'mean'),
        )
        tp['pkg'] = tp.index.str.extract(r'tfp_(curve|fly)_(\w+)')[1].values
        tp['kind'] = tp.index.str.extract(r'tfp_(curve|fly)_')[0].values
        n = len(tp); w = (tp['pnl'] > 0).sum()
        print(f'Trades:     {n}')
        print(f'Winners:    {w} ({w/n:.0%})')
        ww = tp.loc[tp['pnl'] > 0, 'pnl']
        ll = tp.loc[tp['pnl'] <= 0, 'pnl']
        if len(ww): print(f'Avg win:    {ww.mean():>12,.0f}')
        if len(ll): print(f'Avg loss:   {ll.mean():>12,.0f}')
        print(f'Avg hold:   {tp["days"].mean():.0f}d')
        print(f'Total real: {tp["pnl"].sum():>12,.0f}')

        if tp['pkg'].nunique() > 1:
            print('\n--- By Package ---')
            ts = tp.groupby('pkg').agg(
                trades=('pnl', 'count'), total=('pnl', 'sum'),
                avg=('pnl', 'mean'), hit=('pnl', lambda x: f"{(x>0).mean():.0%}"),
                days=('days', 'mean'),
            )
            display(ts)
else:
    print('No closed trades.')

In [ ]:
try:
    tearsheet = QueryBacktestTearSheet.from_backtest(bt)
    plotly_fig = tearsheet.plot_plotly()
    plotly_fig.show()
except Exception as e:
    print(f'Tearsheet: {e}')